# CP1 · Notebook 02 — Detección clásica de carriles

## Objetivo

Construir un pipeline de detección de carriles **sin deep learning**, usando solo técnicas de visión clásica de los 90s. Vas a entender por qué funcionó durante décadas y dónde rompe.

## Pipeline

```
Imagen RGB  ──►  Grayscale  ──►  Gaussian blur  ──►  Canny edges
                                                          │
                                                          ▼
  Overlay  ◄──  Líneas extrapoladas  ◄──  Hough transform  ◄──  ROI mask
```

Cada paso lo implementarás celda a celda. Al final podrás aplicarlo a las 50 imágenes del dataset y categorizar dónde funciona y dónde no.

**Tiempo estimado**: 15 min.
**Requiere**: `01_setup.ipynb` ejecutado correctamente.

## 0. Imports

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time

DATASET_DIR = Path("../datasets/lanes-subset")

# Helper para mostrar imágenes BGR (OpenCV) en matplotlib (RGB)
def show(img, title=None, cmap=None, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4.5))
    if img.ndim == 3:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        ax.imshow(img, cmap=cmap or "gray")
    if title:
        ax.set_title(title)
    ax.axis("off")

## 1. Cargar una imagen 'easy' para empezar

Empezamos por una imagen fácil para entender qué hace cada paso. Más adelante aplicaremos el pipeline a las 50.

In [ ]:
easy_imgs = sorted((DATASET_DIR / "easy").glob("*.png"))
img = cv2.imread(str(easy_imgs[0]))

print(f"Imagen: {easy_imgs[0].name}")
print(f"Shape:  {img.shape}  (H × W × C, BGR)")
show(img, title="Imagen original (easy/0)")

## 2. Paso 1 — Grayscale

El color no nos aporta nada para detectar bordes. Pasamos a escala de grises.

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
show(gray, title="Grayscale")

## 3. Paso 2 — Gaussian blur

Suavizamos para reducir ruido de alta frecuencia antes de buscar bordes.

**Hiperparámetro**: tamaño del kernel. Más grande = más suave = menos detalles. Prueba con 3, 5, 7.

In [ ]:
KERNEL_SIZE = 5

blurred = cv2.GaussianBlur(gray, (KERNEL_SIZE, KERNEL_SIZE), 0)
show(blurred, title=f"Gaussian blur (kernel={KERNEL_SIZE})")

## 4. Paso 3 — Canny edge detection

Canny detecta bordes basándose en el gradiente de intensidad. **Dos hiperparámetros**:
- `low_threshold`: gradiente mínimo para considerarse borde débil
- `high_threshold`: gradiente mínimo para borde fuerte (típicamente ~3× el low)

**Pista**: una regla común es `low=50, high=150`. Pero **para tu CPU y tus imágenes pueden no ser óptimos**. Experimenta y observa.

In [ ]:
LOW_THRESHOLD = 50
HIGH_THRESHOLD = 150

edges = cv2.Canny(blurred, LOW_THRESHOLD, HIGH_THRESHOLD)
show(edges, title=f"Canny edges (low={LOW_THRESHOLD}, high={HIGH_THRESHOLD})")

### 🔎 Experimenta

Modifica los thresholds y vuelve a ejecutar la celda anterior. Observa:
- ¿Qué pasa con `low=20, high=60`?
- ¿Y con `low=100, high=300`?
- ¿Qué configuración mantiene los carriles y reduce ruido?

**Apunta** la configuración que mejor te parezca — la usarás más adelante.

## 5. Paso 4 — ROI mask (Region Of Interest)

Los carriles solo están en la parte inferior y central de la imagen (zona donde nuestro coche "mira"). Aplicamos una máscara trapezoidal para descartar cielo, edificios, etc.

**Nota crítica**: la ROI **asume que la cámara está fija**. Es uno de los puntos más frágiles del enfoque clásico — si la cámara se mueve o el horizonte cambia, esto falla.

In [ ]:
def make_roi_trapezoid(image_shape):
    """Devuelve una máscara binaria con un trapecio en la mitad inferior."""
    h, w = image_shape[:2]
    # 4 vértices del trapecio (en orden: BL, TL, TR, BR)
    polygon = np.array([
        [(int(0.05 * w),    h),                # bottom-left
         (int(0.45 * w),    int(0.60 * h)),    # top-left
         (int(0.55 * w),    int(0.60 * h)),    # top-right
         (int(0.95 * w),    h)]                # bottom-right
    ], dtype=np.int32)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, polygon, 255)
    return mask, polygon[0]


mask, polygon = make_roi_trapezoid(img.shape)
edges_roi = cv2.bitwise_and(edges, mask)

# Visualizar la ROI superpuesta a la imagen original
img_with_roi = img.copy()
cv2.polylines(img_with_roi, [polygon], isClosed=True, color=(0, 255, 0), thickness=3)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
show(img_with_roi, title="ROI superpuesta (verde)", ax=axes[0])
show(edges_roi, title="Edges dentro de la ROI", ax=axes[1])
plt.tight_layout()
plt.show()

## 6. Paso 5 — Hough transform

La Hough transform encuentra **líneas rectas** en una imagen de bordes. Cada píxel "vota" por todas las líneas que podrían pasar por él en el espacio (ρ, θ).

**Hiperparámetros**:
- `threshold`: votos mínimos para considerar una línea válida
- `minLineLength`: longitud mínima en píxeles
- `maxLineGap`: gap máximo entre segmentos que se fusionan

In [ ]:
HOUGH_THRESHOLD = 40
MIN_LINE_LENGTH = 30
MAX_LINE_GAP = 100

lines = cv2.HoughLinesP(
    edges_roi,
    rho=1,                    # resolución espacial en píxeles
    theta=np.pi / 180,        # resolución angular en radianes
    threshold=HOUGH_THRESHOLD,
    minLineLength=MIN_LINE_LENGTH,
    maxLineGap=MAX_LINE_GAP,
)

print(f"Líneas detectadas: {0 if lines is None else len(lines)}")

# Visualizar todas las líneas brutas
overlay_raw = img.copy()
if lines is not None:
    for line in lines:
        x1, y1, x2, y2 = line[0]
        cv2.line(overlay_raw, (x1, y1), (x2, y2), (0, 0, 255), 2)

show(overlay_raw, title="Líneas Hough brutas (rojo)")

## 7. Paso 6 — Separar izquierda/derecha y extrapolar

Tenemos muchas líneas pequeñas. Queremos **2 carriles**: izquierdo (pendiente negativa) y derecho (pendiente positiva — en coordenadas de imagen, y crece hacia abajo).

Agregamos todas las líneas del mismo lado en una sola, promediando pendiente e intercepto.

In [ ]:
def average_lane(lines, side, image_shape, min_slope=0.4):
    """Promedia líneas del lado indicado y extrapola a líneas únicas.
    
    Args:
        lines: array de líneas devuelto por HoughLinesP
        side: 'left' o 'right'
        image_shape: shape de la imagen original
        min_slope: filtro absoluto sobre la pendiente — descarta horizontales
    
    Returns:
        (x1, y1, x2, y2) o None si no hay suficientes líneas
    """
    if lines is None:
        return None
    
    h = image_shape[0]
    slopes, intercepts = [], []
    
    for line in lines:
        x1, y1, x2, y2 = line[0]
        if x2 == x1:
            continue                                # evitar vertical / div 0
        slope = (y2 - y1) / (x2 - x1)
        if abs(slope) < min_slope:
            continue                                # filtra casi-horizontales
        if side == "left" and slope > 0:
            continue
        if side == "right" and slope < 0:
            continue
        intercept = y1 - slope * x1                 # b = y - mx
        slopes.append(slope)
        intercepts.append(intercept)
    
    if not slopes:
        return None
    
    slope = float(np.mean(slopes))
    intercept = float(np.mean(intercepts))
    
    # Extrapolar a y = h (bottom) y y = 0.6*h (top de la ROI)
    y1 = h
    y2 = int(0.6 * h)
    x1 = int((y1 - intercept) / slope)
    x2 = int((y2 - intercept) / slope)
    return (x1, y1, x2, y2)


left_lane  = average_lane(lines, "left",  img.shape)
right_lane = average_lane(lines, "right", img.shape)

print(f"Lane izquierdo: {left_lane}")
print(f"Lane derecho:   {right_lane}")

## 8. Overlay final

In [ ]:
def draw_lanes(image, left, right, color=(0, 255, 255), thickness=8):
    """Dibuja los carriles detectados sobre la imagen."""
    overlay = image.copy()
    for lane in (left, right):
        if lane is not None:
            cv2.line(overlay, lane[:2], lane[2:], color, thickness)
    # mezcla 70/30 para no tapar la imagen
    return cv2.addWeighted(image, 0.7, overlay, 0.3, 0)


result = draw_lanes(img, left_lane, right_lane)
show(result, title="Carriles detectados (overlay amarillo)")

## 9. Empaquetar todo en una función

Hemos construido el pipeline paso a paso. Ahora lo encapsulamos para aplicarlo a muchas imágenes.

In [ ]:
def detect_lanes_classical(
    image,
    canny_low=50,
    canny_high=150,
    blur_kernel=5,
    hough_threshold=40,
    hough_min_length=30,
    hough_max_gap=100,
):
    """Pipeline completo: imagen BGR → imagen BGR con carriles dibujados.
    
    Devuelve (overlay, left_lane, right_lane, elapsed_ms)
    """
    t0 = time.perf_counter()
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (blur_kernel, blur_kernel), 0)
    edges = cv2.Canny(blurred, canny_low, canny_high)
    mask, _ = make_roi_trapezoid(image.shape)
    edges_roi = cv2.bitwise_and(edges, mask)
    lines = cv2.HoughLinesP(
        edges_roi,
        rho=1, theta=np.pi / 180,
        threshold=hough_threshold,
        minLineLength=hough_min_length,
        maxLineGap=hough_max_gap,
    )
    left = average_lane(lines, "left", image.shape)
    right = average_lane(lines, "right", image.shape)
    overlay = draw_lanes(image, left, right)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    return overlay, left, right, elapsed_ms

## 10. Aplicar a las 50 imágenes — grid visual por dificultad

Aquí es donde **vemos dónde rompe el clásico**. Aplicamos el mismo pipeline (con los mismos hiperparámetros) a las 50 imágenes.

In [ ]:
results = {"easy": [], "medium": [], "hard": []}
times_ms = []

for category in results.keys():
    folder = DATASET_DIR / category
    for img_path in sorted(folder.glob("*.png")):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        overlay, left, right, ms = detect_lanes_classical(img)
        results[category].append({
            "name": img_path.name,
            "overlay": overlay,
            "detected": (left is not None, right is not None),
            "time_ms": ms,
        })
        times_ms.append(ms)

print(f"Procesadas {sum(len(v) for v in results.values())} imágenes")
print(f"Tiempo medio por imagen: {np.mean(times_ms):.1f} ms")
print(f"Tiempo total:            {sum(times_ms) / 1000:.1f} s")

In [ ]:
# Grid: 3 muestras de cada categoría
fig, axes = plt.subplots(3, 3, figsize=(15, 9))
for row, category in enumerate(["easy", "medium", "hard"]):
    samples = results[category][:3]
    for col, sample in enumerate(samples):
        ax = axes[row, col]
        ax.imshow(cv2.cvtColor(sample["overlay"], cv2.COLOR_BGR2RGB))
        detected = "L" if sample["detected"][0] else "-"
        detected += "R" if sample["detected"][1] else "-"
        ax.set_title(f"{category} · {sample['name']} · [{detected}] · {sample['time_ms']:.0f}ms")
        ax.axis("off")
plt.tight_layout()
plt.show()

## 11. Tasa de detección por categoría

Una métrica naïve: ¿en cuántas imágenes detectamos ambos carriles (L y R)?

In [ ]:
print(f"{'Categoría':<10} {'N':>4} {'Ambos':>8} {'Solo L/R':>10} {'Ninguno':>10}")
print("-" * 50)
for category, items in results.items():
    n = len(items)
    both = sum(1 for x in items if x["detected"] == (True, True))
    one = sum(1 for x in items if sum(x["detected"]) == 1)
    none_ = sum(1 for x in items if x["detected"] == (False, False))
    print(f"{category:<10} {n:>4} {both:>8} {one:>10} {none_:>10}")

## 12. Reflexión

Antes de pasar al notebook DL, responde mentalmente (no hace falta escribir aquí — guárdalo para `05_preguntas.md`):

1. ¿Qué % de detección obtuviste en `easy`? ¿Y en `hard`? Esperabas esa diferencia?
2. Mira **una imagen `hard` donde fallaste**. ¿Qué hizo fallar al pipeline? (sombras, líneas discontinuas, oclusión, curva muy cerrada, otra cosa?)
3. Si tuvieras que mejorar este pipeline **sin usar deep learning**, ¿qué cambiarías? (espacios de color HSL/LAB, CLAHE, Hough probabilístico vs estándar, polynomial fit, etc.)

## 13. Cierre

Has implementado un detector de carriles clásico funcional. Tiempo de inferencia: **{tiempo_medio} ms en CPU**. Esto es **rapidísimo** — la pregunta es si la calidad basta.

Ve a `03_deep_learning.ipynb` para ver qué hace una red neuronal en las mismas imágenes.

In [ ]:
# Guardar tiempos y resultados para el notebook 04 (comparativa)
import json

summary = {
    "method": "classical_canny_hough",
    "mean_time_ms": float(np.mean(times_ms)),
    "total_images": sum(len(v) for v in results.values()),
    "detection_rate": {
        category: {
            "both": sum(1 for x in items if x["detected"] == (True, True)) / len(items),
            "any":  sum(1 for x in items if any(x["detected"])) / len(items),
        }
        for category, items in results.items()
    },
}

OUT_DIR = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)
with open(OUT_DIR / "02_classical_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("✅ Resumen guardado en outputs/02_classical_summary.json")
print(json.dumps(summary, indent=2))